In [1]:
!pip install --upgrade pip
!pip install datasets transformers evaluate ipywidgets


'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import evaluate


In [9]:
dataset = load_dataset("imdb")
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


In [11]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)


In [12]:
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(1000))


In [13]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="imdb_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)



In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()


C:\Users\RRR\AppData\Local\Temp\ipykernel_18392\957265931.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\RRR\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.310800


c:\Users\RRR\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=750, training_loss=0.23162723541259767, metrics={'train_runtime': 5495.6764, 'train_samples_per_second': 1.092, 'train_steps_per_second': 0.136, 'total_flos': 794804391936000.0, 'train_loss': 0.23162723541259767, 'epoch': 3.0})

In [17]:
trainer.save_model("final_imdb_model")
tokenizer.save_pretrained("final_imdb_model")


('final_imdb_model\\tokenizer_config.json',
 'final_imdb_model\\special_tokens_map.json',
 'final_imdb_model\\vocab.txt',
 'final_imdb_model\\added_tokens.json',
 'final_imdb_model\\tokenizer.json')

In [18]:
from transformers import pipeline
sentiment_model = pipeline("text-classification", model="final_imdb_model")
print(sentiment_model("This movie was absolutely fantastic!"))
print(sentiment_model("I really did not enjoy this movie at all."))


Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.9978135824203491}]
[{'label': 'LABEL_0', 'score': 0.9969221949577332}]
